# $PG(3, 2^k)$ and $AG(3, 2^k)$ arc colorings

This notebook contains a compact, well-structured implementation of:
- A parameterized $GF(2^k)$ factory (recycled from the $PG(2, 2^k)$ notebook).
- Builders for $PG(3, q)$ (projective 3-space points).
- Restriction to $AG(3, q)$ (affine points with $w \neq 0$).
- Coplanarity test via a $4\times 4$ determinant over $GF(2^k)$.
- A configurable MRV + forward-checking coloring solver into caps (no 4 coplanar points per color class).
- The setup and execution for $AG(3, 4)$ ($k=2$, 64 affine points).
- A slow but clear $O(n^4)$ verifier of a cap-proper coloring.

Each major section has a code cell below it. Run cells in order.

## Contents
1. Imports & utilities
2. $GF(2^k)$ factory
3. Projective 3-space builders
4. Coplanarity & utilities
5. Affine extraction & sanity checks
6. Coloring solver (MRV + forward checking)
7. Instance: $AG(3,4)$
8. Verification

In [ ]:
# Imports & small utilities
from itertools import product, combinations
from typing import List, Tuple, Optional, Callable, Dict
import time

Point3 = Tuple[int,int,int,int]  # projective 3-space point (x:y:z:w) in GF(q)^4 \ {0}

## $GF(2^k)$ factory
Recycled from the $PG(2, 2^k)$ notebook. Supports $k=2,3,4$.

In [ ]:
# GF(2^k) factory supporting k=2,3,4 with optional mul-table precompute
def make_field(k: int, irred_poly: Optional[int] = None):
    """Return a small GF object for GF(2^k).
    Representation: integers 0...(2**k-1).
    Default irreducible polys: k=2 -> 0b111 (x^2+x+1), k=3 -> 0b1011 (x^3+x+1), k=4 -> 0b10011 (x^4+x+1).
    """
    class GF:
        def __init__(self, k, irred_poly=None):
            self.k = k
            self.q = 1 << k
            if irred_poly is None:
                defaults = {2:0b111, 3:0b1011, 4:0b10011}
                if k not in defaults:
                    raise ValueError('Default polynomial not provided for this k')
                self.irred = defaults[k]
            else:
                self.irred = irred_poly
            self._mul_table = None
        def add(self, a:int, b:int) -> int:
            return a ^ b
        def mul_no_table(self, a:int, b:int) -> int:
            if a == 0 or b == 0:
                return 0
            res = 0
            for i in range(self.k):
                if (a >> i) & 1:
                    res ^= (b << i)
            top = self.k
            while res >> top:
                shift = (res.bit_length() - 1) - top
                res ^= (self.irred << shift)
            return res & (self.q - 1)
        def mul(self, a:int, b:int) -> int:
            if self._mul_table is not None:
                return self._mul_table[a][b]
            return self.mul_no_table(a,b)
        def pow(self, a:int, e:int) -> int:
            if e == 0:
                return 1
            if a == 0:
                return 0
            res = 1
            base = a
            ee = e
            while ee:
                if ee & 1:
                    res = self.mul(res, base)
                base = self.mul(base, base)
                ee >>= 1
            return res
        def inv(self, a:int) -> int:
            if a == 0:
                raise ZeroDivisionError('0 has no inverse')
            return self.pow(a, self.q - 2)
        def elements(self):
            return list(range(self.q))
        def precompute_mul_table(self):
            table = [[0]*self.q for _ in range(self.q)]
            for i in range(self.q):
                for j in range(self.q):
                    table[i][j] = self.mul_no_table(i,j)
            self._mul_table = table
        def __repr__(self):
            return f'<GF(2^{self.k}) q={self.q} irred=0b{self.irred:b}>'
    return GF(k, irred_poly)

## Projective 3-space builders
Functions to build the points of $PG(3, q)$ using 4-component homogeneous coordinates $(x:y:z:w)$.
We define `normalize_point3`, `build_projective_points3`, and `affine_points_from3`.

In [ ]:
def normalize_point3(pt: Point3, gf) -> Point3:
    """Normalize a projective 3-space point to its canonical representative."""
    x, y, z, w = pt
    for coord in (x, y, z, w):
        if coord != 0:
            inv = gf.inv(coord)
            return tuple(gf.mul(c, inv) for c in pt)
    raise ValueError('zero point')

def build_projective_points3(gf) -> List[Point3]:
    """Build all points of PG(3, q): non-zero homogeneous 4-tuples, one per equivalence class."""
    q = gf.q
    all_pts = set()
    for x, y, z, w in product(range(q), repeat=4):
        if x == 0 and y == 0 and z == 0 and w == 0:
            continue
        all_pts.add(normalize_point3((x, y, z, w), gf))
    return sorted(all_pts)

def affine_points_from3(points: List[Point3]) -> List[Point3]:
    """Extract the affine points of AG(3, q): those with w != 0."""
    return [p for p in points if p[3] != 0]

## Coplanarity & utilities
A $4\times 4$ determinant `det4` and a `coplanar` test.
Four points in $PG(3, q)$ (or $AG(3, q)$) are coplanar if and only if the determinant of the $4\times 4$ matrix formed by their coordinates is zero.
In characteristic 2, cofactor signs are all $+1$, so the expansion simplifies.

In [ ]:
def det3_rows(r0, r1, r2, gf) -> int:
    """3x3 determinant of three 3-tuples over GF(2^k)."""
    x1, y1, z1 = r0
    x2, y2, z2 = r1
    x3, y3, z3 = r2
    t1 = gf.add(gf.mul(x1, gf.mul(y2, z3)), gf.mul(y1, gf.mul(z2, x3)))
    t2 = gf.mul(z1, gf.mul(x2, y3))
    t3 = gf.add(gf.mul(x3, gf.mul(y2, z1)), gf.mul(y3, gf.mul(z2, x1)))
    t4 = gf.mul(z3, gf.mul(x2, y1))
    return gf.add(gf.add(t1, t2), gf.add(t3, t4))

def det4(a: Point3, b: Point3, c: Point3, d: Point3, gf) -> int:
    """4x4 determinant of four projective 3-space points over GF(2^k).
    In characteristic 2 all cofactor signs equal +1, so we simply sum
    a[j] * (3x3 minor removing column j from rows b, c, d).
    """
    rows = [b, c, d]
    result = 0
    for j in range(4):
        minor_rows = [tuple(p[k] for k in range(4) if k != j) for p in rows]
        m = det3_rows(minor_rows[0], minor_rows[1], minor_rows[2], gf)
        result = gf.add(result, gf.mul(a[j], m))
    return result

def coplanar(a: Point3, b: Point3, c: Point3, d: Point3, gf) -> bool:
    """Return True iff the four points lie on a common projective plane."""
    return det4(a, b, c, d, gf) == 0

## Affine extraction & sanity checks
Extract affine points ($w \neq 0$) from $PG(3, q)$ and verify counts:
$|PG(3, q)| = q^3 + q^2 + q + 1$ and $|AG(3, q)| = q^3$.

In [ ]:
def sanity_checks3(gf, points, affine_pts):
    q = gf.q
    expected_proj = q**3 + q**2 + q + 1
    expected_affine = q**3
    assert len(points) == expected_proj, (
        f'unexpected number of projective points: {len(points)} vs {expected_proj}')
    assert len(affine_pts) == expected_affine, (
        f'unexpected number of affine points: {len(affine_pts)} vs {expected_affine}')
    print(f'Sanity checks passed: q={q}, |PG(3,q)|={len(points)}, |AG(3,q)|={len(affine_pts)}')

## Coloring solver (MRV + forward checking)
Adapted from the $PG(2, 2^k)$ notebook. Each color class must be a **cap**:
no four points in the same color class may be coplanar.

Forward checking: when point $i$ is assigned color $c$, every pair $(p, q)$ already
in color class $c$ is paired with $i$ to prune color $c$ from any unassigned point $r$
that would form a coplanar quadruple $\{i, p, q, r\}$.

In [ ]:
def solve_coloring_3d(
    affine_points: List[Point3],
    num_colors: int,
    gf,
    partial_write_path: Optional[str] = None,
    progress_callback: Optional[Callable[[int, int, int], None]] = None,
    timeout: Optional[float] = None
) -> Optional[List[Optional[int]]]:
    """
    Attempt to color `affine_points` with `num_colors` so that no color class
    contains 4 coplanar points.
    Parameters:
      affine_points: list of points with w != 0
      num_colors: number of colors
      gf: field instance
      partial_write_path: optional path to append best partial assignments
      progress_callback: optional callable(depth, nodes_explored, max_depth)
      timeout: optional seconds after which the solver stops and returns None
    Returns: assignment list of length N or None on failure/timeout.
    """
    import sys
    N = len(affine_points)
    assign = [None] * N
    domains = [set(range(num_colors)) for _ in range(N)]
    color_class = {c: [] for c in range(num_colors)}
    change_stack = []
    nodes_explored = 0
    max_depth_reached = 0
    start_time = time.time()

    def default_progress_callback(depth, nodes_explored, max_depth):
        print(f'[Progress] Depth: {depth}, Nodes: {nodes_explored}, Max depth: {max_depth}')
        sys.stdout.flush()

    if progress_callback is None:
        progress_callback = default_progress_callback

    def push(entry):
        change_stack.append(entry)

    def undo(count):
        for _ in range(count):
            typ, data = change_stack.pop()
            if typ == 'assign':
                i, old = data; assign[i] = old
            elif typ == 'domain_remove':
                i, c = data; domains[i].add(c)
            elif typ == 'color_remove':
                i, c = data; color_class[c].remove(i)

    def update_progress(depth):
        nonlocal nodes_explored, max_depth_reached
        nodes_explored += 1
        if depth > max_depth_reached:
            max_depth_reached = depth
            if partial_write_path is not None:
                with open(partial_write_path, 'a') as f:
                    f.write(','.join(str(assign[i]) if assign[i] is not None else '-' for i in range(N)) + '\n')
            if progress_callback is not None:
                progress_callback(depth, nodes_explored, max_depth_reached)

    def select_var():
        best = None; best_size = 10**9
        for i in range(N):
            if assign[i] is None:
                s = len(domains[i])
                if s < best_size:
                    best_size = s; best = i
        return best

    def forward_check(i, color):
        old = assign[i]
        assign[i] = color; push(('assign', (i, old)))
        color_class[color].append(i); push(('color_remove', (i, color)))
        cc = color_class[color]
        # For each pair (q, s) already in the color class (excluding i),
        # check every other point r for coplanarity with {i, q, s, r}.
        for qi in range(len(cc)):
            q = cc[qi]
            if q == i:
                continue
            for si in range(qi + 1, len(cc)):
                s = cc[si]
                if s == i:
                    continue
                pi = affine_points[i]
                pq = affine_points[q]
                ps = affine_points[s]
                for r in range(N):
                    if r == i or r == q or r == s:
                        continue
                    if assign[r] == color:
                        if coplanar(pi, pq, ps, affine_points[r], gf):
                            return False
                    elif assign[r] is None:
                        if color in domains[r]:
                            if coplanar(pi, pq, ps, affine_points[r], gf):
                                domains[r].discard(color)
                                push(('domain_remove', (r, color)))
                                if not domains[r]:
                                    return False
        return True

    def backtrack(depth=0):
        if timeout is not None and (time.time() - start_time) > timeout:
            return False
        update_progress(depth)
        if all(a is not None for a in assign):
            return True
        v = select_var()
        if v is None:
            return True
        domain_snapshot = sorted(domains[v])
        for c in domain_snapshot:
            checkpoint = len(change_stack)
            ok = forward_check(v, c)
            if ok:
                if backtrack(depth + 1):
                    return True
            undo(len(change_stack) - checkpoint)
        return False

    success = backtrack(0)
    if success:
        return assign
    return None

## Instance: $AG(3, 4)$

We build $GF(4)$, construct $PG(3, 4)$ ($85$ projective points), restrict to
$AG(3, 4)$ ($64$ affine points), and color with `num_colors` colors such that
every color class avoids $4$ coplanar points.

In $AG(3, 4)$ each affine plane contains $4^2 = 16$ points, so a color class
can contain at most $3$ points from any plane.  The maximum size of such a
cap is $5$, giving a lower bound of $\lceil 64/5 \rceil = 13$ colors.
The default below uses $15$ colors, for which the solver finds a solution
quickly; change `num_colors` to explore smaller values.

In [ ]:
gf = make_field(2)
gf.precompute_mul_table()  # speed up multiplication
points = build_projective_points3(gf)
affine_pts = affine_points_from3(points)
sanity_checks3(gf, points, affine_pts)

num_colors = 15
print(f'Attempting to color AG(3,4) with {num_colors} colors...')
result = solve_coloring_3d(affine_pts, num_colors, gf, timeout=60)

if result is not None:
    print('Success! Coloring found.')
    import os
    os.makedirs('./results_3d', exist_ok=True)
    with open('./results_3d/affine_ag34_coloring.txt', 'w') as f:
        for pt, color in zip(affine_pts, result):
            x, y, z, w = pt
            inv_w = gf.inv(w)
            ax = gf.mul(x, inv_w)
            ay = gf.mul(y, inv_w)
            az = gf.mul(z, inv_w)
            f.write(f'({ax},{ay},{az}): {color+1}\n')
    color_classes = {c: [] for c in range(num_colors)}
    for idx, color in enumerate(result):
        color_classes[color].append(affine_pts[idx])
    with open('./results_3d/affine_ag34_color_classes.txt', 'w') as f:
        for c in range(num_colors):
            pts = color_classes[c]
            inv_w = [gf.inv(p[3]) for p in pts]
            coords = [(gf.mul(p[0],w), gf.mul(p[1],w), gf.mul(p[2],w)) for p, w in zip(pts, inv_w)]
            f.write(f'Color {c+1}: ' + ', '.join(f'({a},{b},{c_})' for a,b,c_ in coords) + '\n')
    print('Result saved to ./results_3d/affine_ag34_coloring.txt')
    print('Classes saved to ./results_3d/affine_ag34_color_classes.txt')
else:
    print('No coloring found (timeout or impossible).')

## Verification

We implement a slow $O(n^4)$ verifier that checks no color class contains
four coplanar points. Use it to validate a saved coloring file.

In [ ]:
def verify_coloring_file_3d(gf, coloring_file):
    """
    Reads a coloring file of the form (x,y,z): j, groups points by color,
    and checks that no color class contains four coplanar points.
    Returns a dict mapping color -> True (ok) or False (violation found).
    """
    import re
    color_to_points = {}
    with open(coloring_file, 'r') as f:
        for line in f:
            m = re.match(r'\((\d+),(\d+),(\d+)\): (\d+)', line.strip())
            if not m:
                continue
            x, y, z, color = map(int, m.groups())
            # Homogenize: affine (x,y,z) -> projective (x:y:z:1)
            pt = (x, y, z, 1)
            color_to_points.setdefault(color, []).append(pt)
    results = {}
    for color, pts in color_to_points.items():
        ok = True
        for a, b, c, d in combinations(pts, 4):
            if coplanar(a, b, c, d, gf):
                ok = False
                break
        results[color] = ok
    return results


gf_verify = make_field(2)
col_file = './results_3d/affine_ag34_coloring.txt'
results = verify_coloring_file_3d(gf_verify, col_file)
for color, ok in sorted(results.items()):
    status = 'OK (no four coplanar)' if ok else 'FAIL (four coplanar found)'
    print(f'Color {color}: {status}')